In [ ]:
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split,KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

from catboost import CatBoostRegressor

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_file= os.path.join(path,'Q1_data.csv')
df = pd.read_csv(csv_file)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.hist(df['Delivery_Time'],bins=50);

In [ ]:
# Task 1: Write your code here:
df.drop(columns=['Order_ID'],inplace=True)
df.head()

In [ ]:
#part of task 2
df.isna().sum()

In [ ]:
# Task 2: Write your code here:
df= df.dropna()

In [ ]:
df.isna().sum()

In [ ]:
# Task 3: Write your code here:
df= df.drop_duplicates()
df.duplicated().sum()

In [ ]:
# Task 4: Write your code here:
categorical_cols= df.select_dtypes(include=['object']).columns

df= pd.get_dummies(df,columns=categorical_cols,dtype=int)# get_dummies is a function provided by pandas that does oneHot encoding ( iam used to it)
df.head()


In [ ]:
# Task 5: Write your code here:
cols= df.columns.drop('Delivery_Time')
scaler= StandardScaler()
df[cols]= scaler.fit_transform(df[cols])

# df= pd.DataFrame(m,columns= cols)
# df.shape

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X= df.drop(columns='Delivery_Time')
y= df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
kf= KFold(n_splits=5 ,random_state=42,shuffle=True)
model = RandomForestRegressor(n_estimators=100,criterion='absolute_error',random_state=42)

losses=[]
for train_idx,test_idx in kf.split(X,y):
  X_train,y_train = X.iloc[train_idx], y.iloc[train_idx]
  X_test,y_test = X.iloc[test_idx], y.iloc[test_idx]

  model.fit(X_train,y_train)

  y_pred= model.predict(X_test)

  loss= mean_absolute_error(y_test,y_pred)

  losses.append(loss)

print(np.mean(losses))

In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.hist(df['Delivery_Time'],bins=50)

In [ ]:
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100,criterion='absolute_error',random_state=42),
    'CatBoostRegressor': CatBoostRegressor(verbose=0,loss_function='MAE',random_state=42)
}


kf = KFold(n_splits=5, shuffle=True, random_state=42)


model_losses = {}

print("Starting K-Fold Cross-Validation for each model...")

# 4. For each model:
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    fold_losses = [] # List to store loss from each fold

    for fold, (train_index, val_index) in enumerate(kf.split(X,y)):

        X_train,y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_test,y_test = X.iloc[test_idx], y.iloc[test_idx]


        model.fit(X_train, y_train)


        y_pred_proba = model.predict(X_test)


        loss = mean_absolute_error(y_test, y_pred_proba)
        fold_losses.append(loss)

    # Calculate the average loss for the model
    avg_loss = np.mean(fold_losses)
    model_losses[model_name] = avg_loss

    # Print the average cross-validation loss for the current model
    print(f"{model_name} - Average Cross-Validation Loss: {avg_loss:.4f}")

print("\nAll models trained and evaluated. Stored average losses:")
print(model_losses)
print(f'avg MAE = {np.mean(list(model_losses.values()))}')